# 

In [2]:
import numpy as np
import pandas as pd

In [66]:
# copied from as2.py
OPCODES = {
    "nop": 0,
    "+": 1,
    "-": 2,
    "*": 3,
    "/": 4,
    "%": 5,
    ">>": 6,
    "<<": 7,
    "~": 8,
    ">": 9,
    "<": 10,
    ">=": 11,
    "<=": 12,
    "==": 13,
    "!=": 14,
    "jump": 15,
    "jump_imm": 16,
    "call": 17,  # probably don't need this one
    "call_imm": 18,
    "btrue": 19,
    "btrue_imm": 20,
    "bfalse": 21,
    "bfalse_imm": 22,
    "return": 23,
    ";": 23,  # alias
    "load_module": 24,
    "extern_call": 25,
    "loadword": 26,
    "@": 26,  # alias
    "storeword": 27,
    "!": 27,  # alias
    "push_imm": 28,
    "dup": 29,
    "swap": 30,
    "drop": 31,
    "over": 32,
    "rot": 33,
    "dup2": 34,
    "swap2": 35,
    "over2": 36,
    "drop2": 37,  # why?
    "rpush": 38,
    ">r": 38,  # alias
    "rpop": 39,
    "r>": 39,  # alias
    "rcopy": 40,
    "r@": 40,  # alias
    "inc": 41,
    "dec": 42,
    "rcopy2": 43,
    "loadbyte": 44,
    "@b": 44,
    "storebyte": 45,
    "!b": 45,
    "pick": 46,
    "and": 47,
    "or": 48,
    "xor": 49,
    "lsr": 50,
}

# invert the dict
ops = {}
for k, v in op2id.items():
    ops[v] = k

In [74]:
df = pd.read_csv("running_opcodes.csv", names=["opcode"])
df["opcode_name"] = df.apply(lambda r: ops[int(r["opcode"])], axis=1)

In [75]:
df

,opcode,opcode_name
0,18,call_imm
1,28,push_imm
2,26,@
3,28,push_imm
4,25,extern_call
...,...,...
26859,26,@
26860,28,push_imm
26861,25,extern_call
26862,23,;


In [76]:
# create a new df where the columns are values in a rolling window of winsize
def rolling_to_df(df, winsize):
    rol = df.rolling(winsize)
    rows = []
    for window in rol:
        ops = window.reset_index(drop=True)["opcode_name"]
        rows.append(ops)
    return pd.DataFrame(rows)

In [77]:
def markov_count(df, winsize):
    win = rolling_to_df(df, winsize)
    return win.groupby(list(range(winsize)), as_index=False).value_counts().sort_values(by="count", ascending=False)

In [78]:
pd.set_option('display.max_rows', 500)

In [85]:
markov_count(df, 5)

,0,1,2,3,4,count
113,call_imm,call_imm,push_imm,;,*,1057
122,call_imm,push_imm,;,*,push_imm,876
183,push_imm,;,*,push_imm,+,876
23,+,push_imm,+,;,@,800
177,push_imm,+,push_imm,+,;,800
33,;,*,push_imm,+,push_imm,800
11,*,push_imm,+,push_imm,+,800
156,inc,>r,jump_imm,rcopy2,>,664
212,r>,inc,>r,jump_imm,rcopy2,664
76,>r,jump_imm,rcopy2,>,bfalse_imm,664
